# Dense Linear Algebra in NumPy and SciPy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caam37830-sci-comp/notes/blob/main/02_linear_algebra/numpy_scipy_linalg.ipynb)

## Linear Algebra

Linear Algebra is *the* fundamental building block in scientific computing.  Why?  Even if we are dealing with complicated functions, we can always deal with approximations.  If we want to understand a function near a point (sample), the simplest approximation is the *constant* function, which says the function is the same everywhere.  This isn't very interesting.  The next level of approximation is a *linear* approximation, where we model the rate of change near that point. This is *much* more useful, even if it isn't exact, and opens up the possibility of saying all kinds of things about the system that we're studying.

In other words, you can do a lot with degree-1 Taylor series.

Let's first recall some definitions from your mathematics classes:

A **vector space** $V$ over a field $\mathbb{F}$ consists of a set of vectors which are related by addition and scalar multiplication
1. $v, w\in V$ implies $v + w \in V$ (addition)
2. $v \in V, \alpha \in \mathbb{F}$ implies $\alpha v \in V$

These operations are associative and commutative, etc.  Usually what makes things interesting is the existence of a linear map.

A **linear map** $M: V\to W$ is a map from one vector space ($V$) to another ($W$, over the same field $\mathbb{F}$) which satisfies
$M(\alpha v + \beta w) = \alpha M v + \beta M w$

Matrices encode a linear map when we have chosen bases for both $V$ and $W$.

In scientific computing the field $\mathbb{F}$ is typically either the real numbers $\mathbb{R}$ or complex numbers $\mathbb{C}$ (but remember we use floating point arithmetic)

We will also deal with finite dimensional vector spaces. Sometimes these are explicitly vectors in Euclidean space (e.g. in computational geometry), but more often are some sort of function space.


### Householder Notation

Householder notations is standard in numerical linear algebra:
* scalars (elements of $\mathbb{F}$) are denoted with lower-case Greek letters $\alpha, \beta, \dots$
* vectors are denoted with lower-case Latin letters $a, b, \dots, w, v, \dots$
* matrices are denoted with capital letters (Latin or Greek), such as $A, B, \dots$

This was introduced by Householder (see his book "Theory of Matrices in Numerical Analysis", or the modern text "Matrix Computations" by Golub and Van Loan).

### What Do we Compute?

* **basic operations** - e.g. compute $y$ where $y = Ax$
* **solving linear systems** - find $x$ where $Ax = b$.
* **matrix analysis** - understand the action of the matrix $A$ via eigenvalues, SVD, etc.

## Some Places where Linear Algebra Appears

1. Linear regression (example of linear system)
2. Principal components analysis (example of matrix analysis)
3. Solving ODE/PDE (depending on details, involves basic operations and solving linear systems)
4. Data Visualization (often use matrix analysis)
5. Optimization (gradient descent uses basic operations, Newton's method solves a linear system)
6. Imaging - (basic operations, solving systems, some matrix analysis)

And the list goes on.

### Example: Diffusion

One example where linear algebra appears is in measuring the strength of a signal at several sources from receivers.  Consider a simplified problem where we have several points that emit some pollutant.  We aren't able to measure this pollutant directly at the sources.  This can happen for a variety of reasons, such as:
1. The pollutant is completely inaccessible (e.g. far underwater)
2. We're not able to get close for political reasons (people responsible won't cooperate)
3. It is much cheaper to measure from other locations where we already have equipment

And so on.  This problem is related to a variety of other problems in modeling heat or electromagnetic or acoustic signals, and is useful in imaging/remote sensing.  We'll denote the source locations with $x$, and the receiver locations with $y$.  We'll put these in a 2-dimensional plane

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
nx = 3
ny = 3
np.random.seed(0)
x = np.random.rand(nx, 2)
y = np.random.rand(ny, 2)
plt.scatter(x[:,0], x[:,1], label='source')
plt.scatter(y[:,0], y[:,1], label='receiver')
plt.legend()
plt.show()

Each source emits the pollutant at rate $f$.  We represent this with a vector, where `f[i]` is the pollutant emitted by source `i`.

In [ ]:
f_true = np.random.rand(nx)
f_true

We'll assume the pollutant spreads via diffusion.  What follows next might look familiar if you have taken some Physics classes before.  If not, don't worry - the basic idea is that pollution spreads out evenly from each source.  As we get further away from the source, the pollution becomes less concentrated because it is spread out over a larger area.  The function that tells us the concentration of a unit of pollution emitted by the source some distance away is called the *Greens function*.


Let $p(x)$ denote the strength of the pollutant as a function in the 2-dimensional plane.  This means that diffusion obeys the PDE $$\Delta p = f,\quad \Delta := \frac{\partial^2}{\partial x_1^2} + \frac{\partial^2}{\partial x_2^2},  \quad f = \sum_i f_i \delta_{x_i},$$ where $\Delta$ is the laplacian, and $\delta$ is a dirac delta. We're just going to measure the pollutant at the points $y$, where the receivers are located. The amount of pollutant at receiver $j$ from source $i$ is $$- \frac{f_i}{2\pi} \log \| x_i - y_j\|_2$$ (this is the Green's function for $\Delta$).  Because the total amount of pollution at each reciever is the sum of the pollution from each source measured at that receiver, we can write
$$p_j = G_{j,1} f_1 + G{j,2} f_2 + G_{j,3} f_3, \quad G_{j,i} = - \frac{1}{2\pi} \log \| x_i - y_j\|_2,$$
or in matrix vector multiplication
$$p = G f$$
where `p[j]` is the pollution at `y[j]`, `f[i]` is the rate of pollution at source `x[i]`, and `G[j,i]` is $-\frac{1}{2\pi} \log \|x_i - y_j\|_2$

In [ ]:
G = np.empty((ny, nx), dtype=np.float64)
for i in range(nx):      # source
    for j in range(ny):  # receiver
        G[j,i] = -1.0/(2*np.pi) * np.log(np.linalg.norm(x[i] - y[j]))
        
G

Now, we can calculate the concentration of pollution at the receivers using matrix-vector multipication

In [ ]:
p = G @ f_true
p

However, the point is that we want to solve the inverse problem, which is to determine how much each source is polluting from our measurements at the receivers.  In words, we want to find $f$ so that $Gf = p$ for some known $p$.

If you were to write down the solution, you would find $f = G^{-1} p$.  However, you should **never** explicilty invert a matrix on a computer, because you often run into inaccuracy due to floating point representation error. For now, we'll use `np.linalg.solve`

In [ ]:
f_sol = np.linalg.solve(G, p)

print(f'Ground truth: {f_true}')
print(f'Solved: {f_sol}, diff norm: {np.linalg.norm(f_true - f_sol)}')

In [ ]:
G @ f_sol - p

We see that the vector we solved for (`f_sol`) is very close to the true solution of `f_true`, and is close to machine precision.

We are able to accurately obtain the rates at which the sources release pollution based on measurements from some randomly placed sensors.

## Linear Algebra In NumPy

### Basic Operations

Vectors and matrices in NumPy are represented using `np.ndarray`, which we have seen already.  Here is a review:

In [ ]:
n = 3 # dimension of vector space
a, b = 1.5, 2.0 # scalars
x, y = np.random.rand(n), np.random.rand(n)

a*x + b*y # scalar multiplication and vector addition

In [ ]:
m = 4 # dimension of target vector space

A = np.random.randn(m, n) # random linear transformation
z = A @ (a*x + b*y)
z # image

In [ ]:
z2 = a * A @ x + b * A @ y
z2

In [ ]:
z - z2 # difference of image computed in two ways

In [ ]:
k = 3
B = np.random.randn(k, m) # new linear map
z = (B @ A) @ x
z2 = B @ (A @ x)
z - z2

### Solving Linear Systems

Recall from our example that for numerical reasons, you should never invert a matrix to solve a linear system. Let's see an example where we solve `A @ x = b` for `x`

In [ ]:
import numpy.linalg as la
import time

In [ ]:
n = 1000
A = np.random.randn(n,n)
x = np.random.rand(n)
b = A @ x

In [ ]:
t0 = time.time()
Ainv = la.inv(A) # use inversion
x2 = Ainv @ b
t1 = time.time()
print("{} sec.".format(t1 - t0))
la.norm(x - x2)

In [ ]:
t0 = time.time()
x2 = la.solve(A, b) # use solve function
t1 = time.time()
print("{} sec.".format(t1 - t0))
la.norm(x - x2)

We see that you will get a more precise answer, and faster when using `solve` instead of a matrix inverse.

If we want to do a least squares solve: $\min_x \|Ax - b\|$

We can use `lstsq`.  For full-rank (square) matrices, this is equivalent to `solve`.



In [ ]:
x1 = la.solve(A, b)
x2 = la.lstsq(A, b, rcond=None)[0]
la.norm(x1 - x2)

`lstsq` can also be used with over-and under determined linear systems

In [ ]:
m = 1000
n = 500
A = np.random.randn(m, n)
x = np.random.rand(n)
b = A @ x

x2 = la.lstsq(A, b, rcond=None)[0]
la.norm(x - x2)

in the above case, we are able to solve the system exactly because `b` in in the image of `A`

In [ ]:
m = 500
n = 1000
A = np.random.randn(m, n)
x = np.random.rand(n)
b = A @ x

x2 = la.lstsq(A, b, rcond=None)[0]
la.norm(x - x2)

in the above case, there are many possiblities for `x` where `A*x = b`, so we find the smallest-norm solution

## SciPy Linear Algebra

We're now going to switch gears and start using `scipy.linalg` instead of `numpy.linalg`.  From the user's point of view, there isn't really any difference, except `scipy.linalg` has all the same functions as `numpy.linalg` as well as additional functions.  The call signatures are essentially the same, but there are sometimes different implementations under the hood.

* [NumPy linear algebra documentation](https://numpy.org/doc/stable/reference/routines.linalg.html)
* [SciPy linear algebra documentation](https://docs.scipy.org/doc/scipy/reference/linalg.html)

SciPy is built on top of the NumPy `ndarray` data type, so it is fully interoperable.  For example:

In [ ]:
import scipy as sp
import scipy.linalg as sla

n = 1000
A = np.random.randn(n,n)
x = np.random.rand(n)
b = A @ x

x1 = la.solve(A, b) # numpy
x2 = sla.solve(A, b) # scipy
la.norm(x1 - x2)

We'll just import `scipy.linalg` into the `la` namespace, replacing `numpy.linalg`

In [ ]:
import scipy.linalg as la

## Matrix Factorizations

A **matrix factorization** or **matrix decomposition** writes a matrix $A$ as the product of matrices $A = BCD\dots$, where the matrices in the product typically have some special structure.

Here are some examples:
* Diagonal matrices - easy to apply and solve linear systems
* Triangular matrices (upper or lower) - fast to solve linear systems
* Orthonormal matrices: $Q$ orthogonal means $Q^T = Q^{-1}$ (or $Q^T = Q^{\dagger}$ pseudoinverse if $Q$ is not invertible)
* Permutation matrices: sparse orthonormal matrices

Most of the matrix factorizations we will see run in $O(n^3)$ time for a $n\times n$ matrix $A$, or $O(\min(m,n)^2 \max(m,n))$ for a $m\times n$ matrix $A$.

If you want to get really in-depth into how to compute matrix factorizations, take a numerical linear algebra course.  We'll get them treating scipy as a black box (meaning we don't look inside).

### LU decomposition

The first type of factorization we'll look at is a $LU$ decomposition, where $L$ is lower-triangular and $U$ is upper triangular.  For numerical stability, this is often computed with a *pivoting* strategy, which means there is also row or column permutation matrix $P$.
$$ A = PLU$$

In [ ]:
n = 5
A = np.random.randn(n, n)

P, L, U = la.lu(A)

print('L=\n', np.round(L, 3))
print()
print('U=\n', np.round(U, 3))

In [ ]:
la.norm(P @ L @ U - A)

#### Visualizing the factors

We'll plot each matrix as an image, where each entry is a colored square: red is positive, blue is negative, and white is zero. $P$ has a single 1 in each row and column, $L$ is zero above the diagonal (and has ones on the diagonal), and $U$ is zero below the diagonal.

In [ ]:
def plot_matrices(mats, titles, vmax=None):
    """
    plot a list of matrices side by side as images
    zero entries are white, positive entries red, and negative entries blue.
    """
    ratios = [M.shape[1] / M.shape[0] for M in mats]  # make panel widths match matrix shapes
    fig, axs = plt.subplots(1, len(mats), figsize=(2.5*sum(ratios) + len(mats), 3),
                            gridspec_kw={'width_ratios': ratios}, layout='compressed')
    for ax, M, title in zip(axs, mats, titles):
        v = np.abs(M).max() if vmax is None else vmax
        im = ax.imshow(M, cmap='RdBu_r', vmin=-v, vmax=v)
        fig.colorbar(im, ax=ax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.show()

plot_matrices([A, P, L, U], ['$A$', '$P$', '$L$', '$U$'])

The nice thing about triangular matrices is that they can solve linear systems in $O(n^2)$ time, instead of $O(n^3)$ time for general matrices, using the forward or backward substitution algorithms.  There is a special function `solve_triangular` for this reason:

In [ ]:
x = np.random.rand(n)
y = L @ x
x2 = la.solve_triangular(L, y, lower=True)
la.norm(x2 - x)

We can replicate the functionality of `solve` using an LU factorization, using the expression
$$A^{-1} = U^{-1} L^{-1} P^T$$

so that
$$ A^{-1} b = U^{-1} \left( L^{-1} ( P^T b) \right),$$
and we use `solve_triangular` to solve for the matrix inverse.

In [ ]:
def my_solve(A, b):
    """
    solve A * x = b for x
    
    use LU decomposition
    """
    P, L, U = la.lu(A)
    x = la.solve_triangular(
            U,
            la.solve_triangular(
                L,
                P.T @ b,
                lower=True
            ),
            lower=False
        )
    return x

In [ ]:
b = A @ x
x2 = my_solve(A, b)
la.norm(x - x2)

### Exercise

1. Use the `time` module to perform an experiment to verify that `solve_triangular` takes $O(n^2)$ time.  Hint: take LU decompositions of random matrices to get triangular factors of different sizes.

In [ ]:
## Your code here

In [ ]:
import time 
tlist = list()
n  = np.logspace(0,np.log10(5000), num=100) 
n = np.int64(np.round(n))

for i in n:
    A = np.random.randn(i, i)
    x = np.random.rand(i)
    b = A @ x
    P, L, U = la.lu(A)
    y = L @ x

    #Time for solve_triangular
    t0Tri = time.time()
    la.solve_triangular(L, y, lower=True)
    t1Tri = time.time()
    timeTri = t1Tri-t0Tri
    timeTri
    tlist.append(timeTri)

plt.loglog(n, tlist)
plt.xlabel("n ")
plt.ylabel("Time")
plt.show()

2. Suppose $A$ is $n\times n$, so the $LU$ factorization above takes $O(n^3)$ time.  What is a big-O expression for the time to run `my_solve` on $A$? 

The big-O expression for the time to run my_solve on A is $O(n^3) + O(n^2)$. LU factorization takes $O(n^3)$ and each inverse of a triangular matrix takes $O(n^2)$, but two triangular matrices are still $O(n^2)$, and then we sum them up since there is an order performing the algorithm not composed.

### QR decomposition

The $QR$ decomposition, $A = QR$, contains a matrix $Q$ with orthonormal columns, and an upper triangular matrix $R$.  For stability reasons, column pivoting is often used which means there is often a permutation matrix $P$ and $A = QRP$.

In [ ]:
m = 1000
n = 500
A = np.random.randn(m, n)

Q, R = la.qr(A, mode='economic') # no pivoting
la.norm(Q @ R  - A)

In [ ]:
Q.shape, R.shape

In [ ]:
Q, R, P = la.qr(A, pivoting=True) # pivoting
la.norm((Q @ R)  - A[:,P])

#### Visualizing the factors

With `mode='economic'`, a tall $m\times n$ matrix $A$ gives a tall $m \times n$ matrix $Q$ and a square $n\times n$ matrix $R$.  Let's look at a small example.  $R$ is upper triangular, and because the columns of $Q$ are orthonormal, $Q^TQ$ is the identity matrix.  (Note that $QQ^T$ is *not* the identity when $m > n$.)

In [ ]:
A_small = np.random.randn(8, 4)
Q, R = la.qr(A_small, mode='economic')

plot_matrices([A_small, Q, R, Q.T @ Q], ['$A$', '$Q$', '$R$', '$Q^TQ$'])

Geometrically, $Q$ comes from orthogonalizing the columns of $A$ one at a time (the Gram-Schmidt process).  For a $2\times 2$ matrix with columns $a_0, a_1$:
* $q_0$ is $a_0$ normalized to unit length, so $a_0 = r_{00} q_0$
* $q_1$ is the part of $a_1$ perpendicular to $q_0$, normalized to unit length, so $a_1 = r_{01} q_0 + r_{11} q_1$

The entries of $R$ are the coordinates of the columns of $A$ in the orthonormal basis $q_0, q_1$.

In [ ]:
A2 = np.array([[2.0, 1.0],
               [0.5, 1.5]])
Q2, R2 = la.qr(A2)
# QR is only unique up to the signs of the columns of Q.
# Flip signs so that the diagonal of R is positive, as in Gram-Schmidt.
s = np.sign(np.diag(R2))
Q2, R2 = Q2 * s, s[:, None] * R2

fig, ax = plt.subplots(figsize=(5, 5))
arrow = dict(angles='xy', scale_units='xy', scale=1, width=0.012)
ax.quiver([0, 0], [0, 0], A2[0], A2[1], color='C0', label='columns of $A$', **arrow)
ax.quiver([0, 0], [0, 0], Q2[0], Q2[1], color='C1', label='columns of $Q$', **arrow)
# a_1 = r_01 q_0 + r_11 q_1: walk along q_0, then along q_1
p = R2[0, 1] * Q2[:, 0]
ax.plot([0, p[0], A2[0, 1]], [0, p[1], A2[1, 1]], '--', color='gray', zorder=0)
for j in range(2):
    ax.annotate(f'$a_{j}$', A2[:, j], xytext=(5, 5), textcoords='offset points')
    ax.annotate(f'$q_{j}$', Q2[:, j], xytext=(-10, 8), textcoords='offset points')
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 2.5)
ax.set_aspect('equal')
ax.legend(loc='upper left')
plt.show()

print('Q =\n', np.round(Q2, 3))
print('R =\n', np.round(R2, 3))

The QR factorization is used for least-squares solutions, because `Q @ Q.T` projects a vector onto the subspace spanned by the matrix `A`.

\begin{equation}
\begin{aligned}
& \min \| Ax - b \|_2 \\
& = \min \| QR x - b \|_2 \\
& = \min \| Q (R x - Q^T b) \|_2 \\
& = \min \| R x - Q^T b \|_2 
\end{aligned}
\end{equation}

In [ ]:
def my_lstsq(A, b):
    """
    least squares solution ||b - A*x||
    
    Uses QR decomposition
    """
    Q, R = la.qr(A, mode='economic')
    x = la.solve_triangular(R, Q.T @ b, lower=False)
    return x

In [ ]:
x = np.random.rand(n)
b = A @ x

x1 = la.lstsq(A, b)[0]
x2 = my_lstsq(A, b)
la.norm(x1 - x2)

### Excercise

Suppose $A$ is a $n\times n$ matrix, and that the $QR$ factorization takes $O(n^3)$ time.  What is the time to run `my_lstsq` using $A$ in big-O notation?

The computational complexity of QR factorization is $O(n^3)$, and the matirx-vector multiplication "$ Q^{T} b$" costs $ O (n^2)$. Then solving $ Rx = Q^{T}b$ has the cost of solving a triangular system $ O (n^2) $. Thus, the overall computational complexity of `my_lstsq` is ($O(n^3) + O (n^2)$) or just $ O(n^3)$ (because we usually omit the lower orders).

### Eigenvalue Decompositions

A vector $x$ is an eigenvector of $A$ with eigenvalue $\lambda$ if $Ax = \lambda x$.  An eigenvalue decomposition is a decomposition $A = X \Lambda X^{-1}$ where $\Lambda = \operatorname{diag}(\lambda_1, \lambda_2, \dots, \lambda_n)$ is a diagonal matrix.  We can compute such a decomposition using `eig`:

In [ ]:
n = 1000
A = np.random.randn(n, n)
Lam, X = la.eig(A)

columns of `X` are eigenvectors, and eigenvalues are diagonal entries of `Lam`

In [ ]:
x = X[:,0]
la.norm(A @ x - Lam[0] * x)

When `A` is symmetric (or hermitian), there exists and orthonormal basis where every basis element is an eigenvector.  In this case, we can write $A = U\Lambda U^H$.  There is a special function `eigh` for such a situation.

In [ ]:
A = np.random.randn(n,n)
A = A + A.T # make symmetric
Lam, U = la.eigh(A)

In [ ]:
x = U[:,0]
la.norm(A @ x - Lam[0] * x)

In [ ]:
la.norm(U @ U.T - np.eye(n))

#### Visualizing eigenvectors

A general matrix both rotates and stretches vectors.  Eigenvectors are the special directions that only get stretched, since $Ax = \lambda x$.

To see this, we apply a $2\times 2$ matrix to a grid of points, where each point is colored by its starting position so that we can follow where it goes.  Points on the line through an eigenvector (dashed) stay on that line, and are scaled by the eigenvalue.  Here, one eigenvalue is larger than 1, so that direction gets stretched, and the other eigenvalue is negative with magnitude less than 1, so that direction is flipped and shrunk.

In [ ]:
A = np.array([[1.0, 2.0],
              [1.0, 0.5]])
Lam, X = la.eig(A)
Lam = Lam.real  # eig always returns complex eigenvalues, but these are real
print('eigenvalues:', Lam)

# grid of points in the square [-1, 1] x [-1, 1], colored by position
g = np.linspace(-1, 1, 11)
g1, g2 = np.meshgrid(g, g)
pts = np.vstack((g1.ravel(), g2.ravel()))
colors = np.column_stack(((pts[0] + 1)/2, 0.5*np.ones(pts.shape[1]), (pts[1] + 1)/2))

fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
arrow = dict(angles='xy', scale_units='xy', scale=1, width=0.008)
for ax, Y, title in zip(axs, [pts, A @ pts], ['points $x$', 'images $Ax$']):
    ax.scatter(Y[0], Y[1], c=colors, s=12)
    for i in range(2):
        ax.axline((0, 0), X[:, i], color='gray', ls='--', lw=1, zorder=0)
    ax.set_title(title)
    ax.set_aspect('equal')
for i in range(2):
    axs[0].quiver(0, 0, X[0, i], X[1, i], color=f'C{i}', label=f'$x_{i}$', **arrow)
    axs[1].quiver(0, 0, Lam[i]*X[0, i], Lam[i]*X[1, i], color=f'C{i}',
                  label=f'$Ax_{i} = \\lambda_{i} x_{i}$', **arrow)
axs[0].legend(loc='upper left')
axs[1].legend(loc='upper left')
axs[0].set_xlim(-3.2, 3.2)
axs[0].set_ylim(-3.2, 3.2)
plt.show()

Computing eigenvector decompositions takes $O(n^3)$ time for a $n\times n$ matrix.

### Exercises

1. Both `eig` and `eigh` take $O(n^3)$ time.  Which is faster in practice on a symmetric matrix?  Do they produce the same answer on the same symmetric matrix?

As shown in the figure, `eigh` is much faster than `eig` in practice. We checked the eigenvalues produced by both methods (eigenvalues of `eigh` are sorted), they produced the same answer.

2. Write a function that solves a linear system with a symmetric matrix using the Eigenvalue decomposition.  Is this function faster or slower than using LU in `my_solve`?

As shown in the figure, `my_solve` is faster than `eigh` in practice. 

In [ ]:
## Your code here


In [ ]:
## Your code here
'''
1. Compare the speed and difference of eig & eigh
'''
n = np.logspace(0,np.log10(1000), num=100) 
n = np.int64(np.round(n))
teig=list()
teigh=list()
valdiff = list()
for i in n:
    A = np.random.randn(i,i)
    A = A + A.T
    t0eig = time.time()
    Lam1, X = la.eig(A)
    t1eig = time.time()
    teigdiff = t1eig-t0eig
    teig.append(teigdiff)
    
    t0eigh = time.time()
    Lam2, U = la.eigh(A)
    t1eigh = time.time()
    teighdiff = t1eigh-t0eigh
    teigh.append(teighdiff)
    
    valdiff.append(np.max(np.abs(np.sort(Lam1) - np.sort(Lam2))))

In [ ]:
'''
1. Time comparison
'''
plt.loglog(n, teig, label='eig')
plt.loglog(n, teigh, label='eigh')
plt.xlabel("n")
plt.ylabel("Time")
plt.legend(loc='best')
plt.show()

'''
1. Difference
'''
np.max(valdiff)

In [ ]:
'''
2. Compare the speed of EVD v.s. LU when solving linear systems
'''
def solve_eigh(A, b):
    Lam, U = la.eigh(A)
    Ainv = U @ np.diag(1/Lam) @ U.T
    x = Ainv@b
    return(x)


n = np.logspace(0,np.log10(1000), num=100) 
n = np.int64(np.round(n))
tlu=list()
teigh=list()
valdiff = list()
for i in n:
    A = np.random.randn(i,i)
    A = A + A.T
    x = np.random.rand(i)
    b = A @ x
    t0lu = time.time()
    x1 = my_solve(A, b)
    t1lu = time.time()
    tludiff = t1lu-t0lu
    tlu.append(tludiff)
    
    t0eigh = time.time()
    x2 = solve_eigh(A,b)
    t1eigh = time.time()
    teighdiff = t1eigh-t0eigh
    teigh.append(teighdiff)

In [ ]:
'''
2. Time comparison
'''
plt.loglog(n,tlu, label='lu')
plt.loglog(n,teigh, label='eigh')
plt.xlabel("n")
plt.ylabel("Time")
plt.legend(loc='best')
plt.show()

### SVD

The singular value decomposition is an extremely useful practical and theoretical tool.  We can decompose a $m\times n$ matrix $A$ as $A = U \Sigma V^T$, where $U$ is a $m \times m$ matrix with orthonormal columns (called left singular vectors), $V$ is a $n\times n$ matrix with orthonormal columns (called right singular vectors), and $\Sigma$ is a diagonal matrix with positive entries decreasing in magnitude (called singular values).

The top singular value solves the variational problem $\sigma_0 = \max u^T A v$ subject to $\|u\|_2 = 1, \|v\|_2=1$, and describes the direction in which $A$ induces the largest change in maginitude in a vector. The next singular value is defined similarly on the subspaces orthogonal to $u$ and $v$, and so on.

In [ ]:
n = 4
A = np.random.randn(n,n)
U, S, Vt = la.svd(A)

In [ ]:
plot_matrices([A, U, np.diag(S), Vt], ["$A$", "$U$", "$S$", "$V^T$"])

One way to visualize the action of a matrix is seeing how it maps the unit sphere.  The image is an ellipsoid: the left singular vectors (columns of $U$) give the directions of its axes, and the singular values give the lengths of these axes.  The right singular vectors (columns of $V$) are the points on the sphere that get mapped onto these axes, $A v_i = \sigma_i u_i$.

In [ ]:
A = np.random.randn(2,2)
U, S, Vt = la.svd(A)

theta = np.linspace(0, 2*np.pi, num=200, endpoint=True)
xx = np.vstack((np.cos(theta), np.sin(theta)))
yy = A @ xx

fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111)
ax.plot(yy[0], yy[1])
ax.set_aspect('equal')
ax.scatter(S*U[0], S*U[1])
plt.show()

We can break this down further.  Reading $Ax = U\Sigma V^T x$ from right to left, every matrix acts in three steps:
1. $V^T$ rotates (or reflects) the circle, taking the right singular vectors $v_i$ to the coordinate axes
2. $\Sigma$ stretches the $i$-th coordinate axis by $\sigma_i$, turning the circle into an ellipse aligned with the axes
3. $U$ rotates (or reflects) the ellipse, so its axes point along the left singular vectors $u_i$

#### SVD as a sum of rank-1 matrices

Writing out the matrix product, the SVD expresses $A$ as a weighted sum of rank-1 matrices (outer products of singular vectors):
$$A = U\Sigma V^T = \sum_i \sigma_i u_i v_i^T$$

Here is a small example: first the factors, then the rank-1 terms of the sum, all on the same color scale as $A$.  Because the singular values decrease, the first terms make up most of $A$.

In [ ]:
A = np.array([[0, 1, 2, 3, 4],
              [5, 4, 3, 2, 1],
              [1, 3, 2, 4, 3]], dtype=float)
U, S, Vt = la.svd(A, full_matrices=False)
print('singular values:', S)

plot_matrices([A, U, np.diag(S), Vt], ['$A$', '$U$', '$\\Sigma$', '$V^T$'])

terms = [S[i] * np.outer(U[:, i], Vt[i]) for i in range(len(S))]
plot_matrices(terms, [f'$\\sigma_{i} u_{i} v_{i}^T$' for i in range(len(S))],
              vmax=np.abs(A).max())

#### Low-rank approximation

Keeping just the first $k$ terms of the sum gives a rank-$k$ matrix
$$A_k = \sum_{i < k} \sigma_i u_i v_i^T.$$
This is the *best* rank-$k$ approximation to $A$, with error $\|A - A_k\|_2 = \sigma_{k}$.  Storing $A_k$ only requires $k(m + n + 1)$ numbers instead of $mn$, so when the singular values decay quickly we can compress $A$ a lot with little error.

A grayscale image is just a matrix of pixel intensities, so let's try this on a picture.  Matplotlib comes with a sample photo of the computer scientist Grace Hopper.

In [ ]:
import matplotlib.cbook as cbook

img = plt.imread(cbook.get_sample_data('grace_hopper.jpg', asfileobj=False))
img = img.mean(axis=2)  # average the red, green, and blue channels to get grayscale
m, n = img.shape

U, S, Vt = la.svd(img, full_matrices=False)

plt.semilogy(S)
plt.xlabel('$i$')
plt.ylabel('$\\sigma_i$')
plt.title('singular values of the image')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 5))
for ax, k in zip(axs, [5, 20, 50]):
    img_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k]
    ax.imshow(img_k, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'rank {k}: {k*(m + n + 1) / (m*n):.0%} of the storage')
axs[-1].imshow(img, cmap='gray', vmin=0, vmax=255)
axs[-1].set_title(f'original: rank {min(m, n)}')
for ax in axs:
    ax.axis('off')
plt.show()

In [ ]:
k = 20
img_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k]
la.norm(img - img_k, 2) - S[k]  # the error is exactly sigma_k

### Exercises

1. Computing the SVD takes $O(n^3)$ time for a $n\times n$ matrix, just like all the other matrix factorizations we've seen.  Make plots of the runtimes of `svd`, `eigh`, `eig`, `qr` and `lu` as we increase $n$.  Which is fastest in practice?

From the plot, we see that from slowest to fastest is: `eig, svd, eigh, qr, lu`. `lu` is fastest in practice.

2. Write a function that solves a linear system using the Singular Value Decomposition.
3. What is the SVD of a Hermitian (symmetric matrix)?  Is `svd` or `eigh` faster?

From the last figure, we can see that `eigh` is faster on a Hermitian.

In [ ]:
## Your code here


In [ ]:
'''
1. Time comparison of SVD, eig, eigh, QR, LU
'''
n = np.logspace(0,np.log10(1000), num=100) 
n = np.int64(np.round(n))
tsvd=list()
teigh=list()
teig=list()
tqr=list()
tlu=list()

for i in n:
    A = np.random.randn(i,i)

    t0 = time.time()
    U, S, Vt = la.svd(A)
    t1 = time.time()
    tsvd.append(t1-t0)
    
    t0 = time.time()    
    Lam, U = la.eig(A)
    t1 = time.time()
    teig.append(t1-t0)
    
    t0 = time.time()    
    Lam, U = la.eigh(A)
    t1 = time.time()
    teigh.append(t1-t0)
    
    t0 = time.time()    
    Q, R = la.qr(A, mode='economic') # no pivoting
    t1 = time.time()
    tqr.append(t1-t0)
    
    t0 = time.time()    
    P, L, U = la.lu(A)
    t1 = time.time()
    tlu.append(t1-t0)

In [ ]:
'''
1. Time comparison of SVD, eig, eigh, QR, LU
'''
plt.loglog(n, teig, label='eig')
plt.loglog(n, tsvd, label='svd')
plt.loglog(n, teigh, label='eigh')
plt.loglog(n, tqr, label='qr')
plt.loglog(n, tlu, label='lu')
plt.xlabel("n")
plt.ylabel("Time")
plt.legend(loc='best')
plt.show()

In [ ]:
'''
2. Define a function that solves a linear system using SVD
'''
def solve_svd(A, b):
    U, S, Vt = la.svd(A)
    Ainv = U@np.diag(1/S)@Vt
    x = Ainv@b
    return(x)

In [ ]:
'''
3. Compare svd and eigh on symmetric matrices
'''

n = np.logspace(0,np.log10(2000), num=100) 
n = np.int64(np.round(n))
tsvd=list()
teigh=list()

for i in n:
    A = np.random.randn(i,i)
    A = A + A.T
    t0 = time.time()
    U, S, Vt = la.svd(A)
    t1 = time.time()
    tsvd.append(t1-t0)

    t0 = time.time()    
    Lam, U = la.eigh(A)
    t1 = time.time()
    teigh.append(t1-t0)
    

In [ ]:
plt.loglog(n, tsvd, label='svd')
plt.loglog(n, teigh, label='eigh')
plt.xlabel("n")
plt.ylabel("Time")
plt.legend(loc='best')
plt.show()

### Principal Component Analysis

Principal component analysis (PCA) finds the directions in which a data set varies the most.  Suppose we have $m$ data points in $\mathbb{R}^n$, stored as the rows of a $m\times n$ matrix $X$.  After subtracting the mean of each column (centering the data), the *covariance matrix* is
$$C = \frac{1}{m-1} X^T X.$$
$C$ is symmetric, so it has an orthonormal basis of eigenvectors.  These eigenvectors are the *principal components*, and the eigenvalues are the variance of the data in the direction of each principal component.

If $X = U\Sigma V^T$, then
$$C = \frac{1}{m-1} V \Sigma U^T U \Sigma V^T = V \frac{\Sigma^2}{m-1} V^T,$$
so the right singular vectors of $X$ are the principal components, and $\sigma_i^2 / (m-1)$ are the variances.  This is how PCA is usually computed in practice, since forming $X^TX$ can lose accuracy in floating point.

Let's generate some correlated 2D data: we stretch independent Gaussian samples along the axes, then rotate and shift them.

In [ ]:
m = 500
theta = np.pi / 6
Rot = np.array([[np.cos(theta), -np.sin(theta)],
                [np.sin(theta),  np.cos(theta)]])
X = (np.random.randn(m, 2) * [2.0, 0.6])  @ Rot.T + [2.0, 1.0]

xbar = X.mean(axis=0)
Xc = X - xbar  # center the data

# PCA from the eigenvalue decomposition of the covariance matrix
C = Xc.T @ Xc / (m - 1)
Lam, W = la.eigh(C)
Lam, W = Lam[::-1], W[:, ::-1]  # eigh sorts eigenvalues in increasing order; reverse

# PCA from the SVD of the centered data
U, S, Vt = la.svd(Xc, full_matrices=False)

print('variances from eigh:', Lam)
print('variances from svd: ', S**2 / (m - 1))
print('|W^T V| =\n', np.abs(W.T @ Vt.T))  # identity: same vectors, up to sign

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
arrow = dict(angles='xy', scale_units='xy', scale=1, width=0.008)

axs[0].scatter(X[:, 0], X[:, 1], s=8, alpha=0.4, color='gray')
for i in range(2):
    # arrow of length two standard deviations along the i-th principal component
    d = 2 * np.sqrt(Lam[i]) * W[:, i]
    axs[0].quiver(*xbar, *d, color=f'C{i}', label=f'PC {i}', **arrow)
axs[0].set_xlabel('$x[0]$')
axs[0].set_ylabel('$x[1]$')
axs[0].set_title('data and principal components')
axs[0].legend(loc='upper left')

Z = Xc @ Vt.T  # coordinates of the data in the principal component basis
axs[1].scatter(Z[:, 0], Z[:, 1], s=8, alpha=0.4, color='gray')
axs[1].set_xlabel('PC 0')
axs[1].set_ylabel('PC 1')
axs[1].set_title('data in principal component coordinates')

for ax in axs:
    ax.set_aspect('equal')
plt.show()

In the principal component coordinates the data is centered, and the two coordinates are uncorrelated.  For higher-dimensional data, plotting the first two principal components is a common way to visualize the data, since these are the two directions that capture the most variance.  Dropping the remaining components is a form of dimensionality reduction, which is the same idea as the low-rank approximation above: $X_k = U_k \Sigma_k V_k^T$ is the best rank-$k$ approximation to the centered data.